## <h2>String Matching Utilities for Name Comparison and Record Linkage</h2>

This note provides a collection of Python utility functions designed to perform *robust string comparison and matching*, particularly for use cases involving *names*, *person records*, or other *textual identifiers*. It is especially useful in contexts like:

* Data cleaning and preprocessing
* Record linkage / entity resolution
* Fuzzy matching of names across datasets
* Deduplication tasks
* Natural language preprocessing

Although originally developed for personal projects, this code is structured to be reusable and extensible in a variety of real-world applications.

### <h3>What This Script Does</h3>

It includes three core functions:

#### <h4>1. `str_normalize`</h4>

A flexible *string normalization* function that:

* Converts text to lowercase
* Removes or replaces specified punctuation or characters
* Strips diacritics (e.g., "José" → "jose")
* Collapses extra whitespace

**Use case:** Prepares strings for consistent comparison, especially useful when matching messy real-world names or textual data.

#### <h4>2. `name_distance`</h4>

Computes a *distance score* between two names using metrics such as:

* Levenshtein distance
* Damerau-Levenshtein distance
* Hamming distance
* Jaro similarity

It optionally:

* Normalizes input strings
* Reorders words for optimal alignment (via Hungarian algorithm)
* Penalizes unmatched words

**Use case:** Accurately compare two names (e.g., `"Robert Kennedy Jr."` vs `"Kennedy, Robert Jr"`) even when formatting or order varies.

#### <h4>3. `match_strings`</h4>

Performs optimal *string matching* between two lists using the selected distance metric. It:

* Normalizes strings (optional)
* Calculates pairwise distances
* Applies optimal assignment (Hungarian algorithm)
* Allows distance threshold (`cutoff`) to exclude poor matches
* Returns either matched values or index pairs

**Use case:** Map names or identifiers from two sources (e.g., customer lists, bibliographic records, etc.), even if spelling or formatting differs.

### <h3>Example Use Cases</h3>

* Matching author names across bibliographic datasets
* Linking names in government/public records
* Fuzzy merging of survey respondent names
* Data deduplication for CRM systems


### <h3>Script</h3>

Here I collect the functions and packages necessary for performing the tasks mentioned above. Users of Google Colab may need to run the following code in order to be able load text distance functions.

```python
# Packages not in Colab
!pip install textdistance
```

In [2]:
# Importing packages
from itertools import product
import numpy as np
import pandas as pd
import re
from scipy.optimize import linear_sum_assignment
import textdistance
import unicodedata
from typing import List, Tuple, Literal, Optional, Union, Iterable

# String normalizer
def str_normalize(
    s: str,
    remove: Optional[Union[str, Iterable[str]]] = None,
    replace_with_space: bool = True
) -> str:
    """
    Normalize a string.

    Normalize a string by lowercasing it, removing or replacing specified characters, replacing diacritics with ASCII equivalents, and reducing multiple spaces to a single space.

    Parameters
    ----------
    s : str
        Input string to be normalized.

    remove : str or iterable of str, optional
        Characters to remove or replace from the string.
        Can be a single string (e.g. ".'-") or a list/tuple of characters.

    replace_with_space : bool, default=True
        If True, replaces each character in `remove` with a space.
        If False, characters are removed entirely.

    Returns
    -------
    str
        The normalized string.

    Examples
    --------
    >>> str_normalize("Shaqüille O'Neil Jr.", remove = "'.,", replace_with_space = True)
    'shaquille o neil jr'
    """

    # Check string argument
    if not isinstance(s, str):
        raise TypeError("First argument must be a string")

    # Normalize `remove` parameter
    if remove is not None:
        if isinstance(remove, str):
            remove_chars = set(remove)
        elif isinstance(remove, Iterable) and all(isinstance(c, str) for c in remove):
            remove_chars = set("".join(remove))
        else:
            raise TypeError("`remove` must be a string or an iterable of strings")
    else:
        remove_chars = set()

    # Convert to lowercase
    s = s.lower()

    # Remove or replace specified characters
    if remove_chars:
        for char in remove_chars:
            s = s.replace(char, " " if replace_with_space else "")

    # Remove diacritics
    s = unicodedata.normalize("NFKD", s)
    s = (s.encode(encoding = "ascii", errors = "ignore")
          .decode(encoding = "ascii"))

    # Collapse multiple spaces and strip
    s = re.sub(r'\s+', ' ', s).strip()

    return s

# Name distance
def name_distance(
    name1: str,
    name2: str,
    metric: str = "levenshtein",
    normalize: bool = True,
    remove: Optional[Union[str, Iterable[str]]] = None,
    replace_with_space: bool = True,
    reorder: bool = True
) -> float:
    """
    Compute the string distance between two names using a chosen metric.

    Supports normalization (case folding, character removal, diacritics stripping), and optional word reordering using optimal assignment.

    Parameters
    ----------
    name1 : str
        First name or string to compare.

    name2 : str
        Second name or string to compare.

    metric : {'levenshtein', 'damerau_levenshtein', 'hamming', 'jaro'}, default='levenshtein'
        String distance metric to use.

    normalize : bool, default=True
        Whether to normalize names before comparison.

    remove : str or iterable of str, optional
        Characters to remove or replace in normalization.

    replace_with_space : bool, default=True
        If True, replaces removed characters with space. If False, removes them.

    reorder : bool, default=True
        If True, treats names as bags of words and finds optimal word alignment.

    Returns
    -------
    float
        The total distance between the names. Lower is more similar.

    Examples
    --------
    >>> name_distance('Kennedy Robert Jr', 'Robert F. Kennedy Jr.',
    ...               metric='levenshtein', normalize=True, remove=[".", "-", "'"])
    2.0
    """

    # Input validation
    if not isinstance(name1, str) or not isinstance(name2, str):
        raise TypeError("Both name1 and name2 must be strings.")

    if not isinstance(metric, str):
        raise TypeError("`metric` must be a string.")

    metric = metric.lower()
    metric_funcs = {
        'levenshtein': textdistance.levenshtein.distance,
        'damerau_levenshtein': textdistance.damerau_levenshtein.distance,
        'hamming': textdistance.hamming.distance,
        'jaro': textdistance.jaro.distance
    }

    if metric not in metric_funcs:
        raise ValueError(f"Unsupported metric '{metric}'. Choose from: {list(metric_funcs)}")

    distance_func = metric_funcs[metric]

    # Normalization
    if normalize:
        name1 = str_normalize(name1, remove = remove,
                              replace_with_space = replace_with_space)
        name2 = str_normalize(name2, remove = remove,
                              replace_with_space = replace_with_space)

    # Word splitting
    words1 = name1.split() if reorder else [name1]
    words2 = name2.split() if reorder else [name2]

    # Cost matrix
    cost_matrix = np.full((len(words1), len(words2)),
                          fill_value = max(len(name1), len(name2)))

    for i, w1 in enumerate(words1):
        for j, w2 in enumerate(words2):
            try:
                cost_matrix[i, j] = distance_func(w1, w2)
            except Exception:
                # Handle distance function limitations (e.g., hamming requires equal length)
                cost_matrix[i, j] = max(len(w1), len(w2))

    # Optimal assignment
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    matching_dist = sum(cost_matrix[i, j] for i, j in zip(row_ind, col_ind))

    # Unmatched words penalty
    unmatched_dist = 0
    if len(words1) > len(words2):
        unmatched_words = [words1[i] for i in range(len(words1))
                           if i not in row_ind]
        unmatched_dist = len(" ".join([word for word in unmatched_words]))
    elif len(words2) > len(words1):
        unmatched_words = [words2[j] for j in range(len(words2))
                           if j not in col_ind]
        unmatched_dist = len(" ".join([word for word in unmatched_words]))

    return float(matching_dist + unmatched_dist)

# Match strings
def match_strings(
    strlist1: Iterable[str],
    strlist2: Iterable[str],
    metric: str = "levenshtein",
    cutoff: Optional[Union[int, float]] = None,
    indices: bool = False,
    normalize: bool = True,
    remove: Optional[Union[str, Iterable[str]]] = None,
    replace_with_space: bool = True,
    reorder: bool = True
) -> List[Tuple[Union[int, str], Union[int, str]]]:
    """
    Match elements from two lists of strings using pairwise string distance and optimal assignment.

    Parameters
    ----------
    strlist1 : Iterable[str]
        First list of strings.

    strlist2 : Iterable[str]
        Second list of strings.

    metric : {'levenshtein', 'damerau_levenshtein', 'hamming', 'jaro'}, default='levenshtein'
        String distance metric to use.

    cutoff : int or float, optional
        String distance cutoff below which strings can be matched.

    indices : bool, default=False
        If True, return index pairs instead of string pairs.

    normalize : bool, default=True
        Whether to normalize strings before comparison.

    remove : str or iterable of str, optional
        Characters to remove/replace in normalization.

    replace_with_space : bool, default=True
        Whether to replace removed characters with a space.

    reorder : bool, default=True
        Whether to align words optimally within names.

    Returns
    -------
    List[Tuple[int, int]] or List[Tuple[str, str]]
        List of matched pairs (with unmatched items paired with None or empty string).

    Example
    -------
    >>> match_strings(['tsar', 'fry', 'rice', 'fake', 'gray'],
                      ['lice', 'czar', 'fly', 'bake'],
                      metric = 'Levenshtein', normalize = True,
                      remove = ['.', '-', "'"])
    """

    # Input validation
    if not (all([isinstance(s, str) for s in strlist1]) and
            all([isinstance(s, str) for s in strlist2])):
        raise TypeError("Both strlist1 and strlist2 must contain only strings.")

    if not isinstance(metric, str):
        raise TypeError("`metric` must be a string.")

    metric = metric.lower()
    metric_funcs = {
        'levenshtein': textdistance.levenshtein.distance,
        'damerau_levenshtein': textdistance.damerau_levenshtein.distance,
        'hamming': textdistance.hamming.distance,
        'jaro': textdistance.jaro.distance
    }

    if metric not in metric_funcs:
        raise ValueError(f"Unsupported metric '{metric}'. Choose from: {list(metric_funcs)}")

    if cutoff is not None:
        if not isinstance(cutoff, (int, float)):
            raise TypeError("`cutoff` must be an int, float, or None.")
        if cutoff < 0:
            raise ValueError("`cutoff` must be non-negative.")

    # Normalization
    if normalize:
        names1 = [str_normalize(name1, remove = remove,
                                replace_with_space = replace_with_space)
                  for name1 in strlist1]
        names2 = [str_normalize(name2, remove = remove,
                                replace_with_space = replace_with_space)
                  for name2 in strlist2]
    else:
        names1 = strlist1
        names2 = strlist2

    # Distance function for normalized strings
    def str_dist(str1, str2):
        return name_distance(str1, str2, metric = metric,
                             normalize = False, remove = None,
                             replace_with_space = True,
                             reorder = reorder)

    # Build distance matrix
    n1, n2 = len(names1), len(names2)
    max_len = max((len(s) for s in names1 + names2))
    dist_matrix = np.full((n1, n2), fill_value = max_len)

    for i, name1 in enumerate(names1):
        for j, name2 in enumerate(names2):
            dist_matrix[i, j] = str_dist(name1, name2)

    # Optimal matching
    max_dist = max_len if cutoff is None else cutoff
    row_ind, col_ind = linear_sum_assignment(dist_matrix)
    matched_pairs = [(i, j) for i, j in list(zip(row_ind, col_ind))
                     if dist_matrix[i, j] <= max_dist]

    # Add unmatched pairs
    matched_rows = set(i for i, j in matched_pairs)
    matched_cols = set(j for i, j in matched_pairs)

    unmatched_rows = [(i, None) for i in range(n1) if i not in matched_rows]
    unmatched_cols = [(None, j) for j in range(n2) if j not in matched_cols]

    all_pairs = matched_pairs + unmatched_rows + unmatched_cols

    # Format output
    if indices:
        return all_pairs

    result = []
    for i, j in all_pairs:
        s1 = strlist1[i] if i is not None else ""
        s2 = strlist2[j] if j is not None else ""
        result.append((s1, s2))

    # Sort result for consistent order
    result.sort(key = lambda x: x[0] or x[1])

    return result

### <h3>Application</h3>

Here I provide a small application that illustrates the use of the functions introduced above.

We have two lists of city names, where some names are common to both lists and other names may not be. In addition, common names sometimes happen to be  written slightly different across these two lists. These two list are presented in the two tables below.

</br>

Table 1: List A
| Index | City Name        |
| ----- | ---------------- |
| 1     | Silverbrook      |
| 2     | North Elmsfield  |
| 3     | Lakewood Heights |
| 4     | Eastmoor         |
| 5     | Cresthill        |
| 6     | New Averton      |
| 7     | Fairgrove        |
| 8     | Grand Rivertown  |
| 9     | Maple Bay        |
| 10    | Hollow Creek     |

</br>

Table2: List B
| Index | City Name        |
| ----- | ---------------- |
| 1     | Grand-River Town |
| 2     | Silverbrook      |
| 3     | Windmere         |
| 4     | Fairgrove        |
| 5     | North-Elmsfield  |
| 6     | Pine Hollow      |
| 7     | New Averton      |
| 8     | Lakewood Hts.    |
| 9     | Crest Hill       |
| 10    | Eastmoore        |

</br>

The code below provides a match of the names in the two lists.


In [3]:
# List A – Original City Names
strlist1 = [
    "New Averton",
    "Crest Hill",
    "Silverbrook",
    "Eastmoor",
    "Lakewood Heights",
    "Pine Hollow",
    "Fairgrove",
    "Windmere",
    "Oakridge Bay",
    "Maple Grove"
]

# List B – Shuffled and Slightly Altered City Names
strlist2 = [
    "Grand-River Town",
    "Silverbrook",
    "Windmere",
    "Fairgrove",
    "North-Elmsfield",
    "Pine Hollow",
    "New Averton",
    "Lakewood Hts.",
    "Crest Hill",
    "Eastmoore"
]

# Matching strings
str_matches = match_strings(strlist1, strlist2, metric = "levenshtein",
                            cutoff = 5, normalize = True, reorder = True,
                            remove = ["-", ".", "'"])
str_matches = [(n1, n2, name_distance(n1, n2, remove = ["-", ".", "'"]))
               for n1, n2 in str_matches]

# Result
lstr = max([len(name) for name in strlist1 + strlist2]) + 1
lnum = len('distance')
header = ['List A', 'List B', 'Distance']
table = ['| {0:<{3}} | {1:<{3}} | {2:<{4}} |\n'.format(*header, lstr, lnum),
         '|:{0}|:{0}|:{1}|\n'.format('-'*(lstr + 1), '-'*(lnum + 1))]
for i in range(len(str_matches)):
    row = '|'.join([f'{str_matches[i][0]:<{lstr}} ',
                    f' {str_matches[i][1]:<{lstr}} ',
                    f'{str_matches[i][2]:>{lnum}} '])
    row = f'| {row} |\n'
    table.append(row)
print(''.join(table))

| List A            | List B            | Distance |
|:------------------|:------------------|:---------|
| Crest Hill        | Crest Hill        |     0.0  |
| Eastmoor          | Eastmoore         |     1.0  |
| Fairgrove         | Fairgrove         |     0.0  |
|                   | Grand-River Town  |    16.0  |
| Lakewood Heights  | Lakewood Hts.     |     4.0  |
| Maple Grove       |                   |    11.0  |
| New Averton       | New Averton       |     0.0  |
|                   | North-Elmsfield   |    15.0  |
| Oakridge Bay      |                   |    12.0  |
| Pine Hollow       | Pine Hollow       |     0.0  |
| Silverbrook       | Silverbrook       |     0.0  |
| Windmere          | Windmere          |     0.0  |

